# CatBoost + MLP + LightGBM 3-way 블렌드 (Colab, 전체 데이터)

목표: 기존 CatBoost+MLP 2-way 블렌드(로컬 851.19 / 실제 대시보드 924)에 튜닝된 LightGBM을 3번째 멤버로 추가해서 더 오를 수 있는지 확인합니다.

## 사전 준비 (이 셀 실행 전에 직접 해야 함)
1. Colab 상단 메뉴 런타임 > 런타임 유형 변경 > **GPU** 선택 (T4로도 충분)
2. 로컬 `data/train.csv`, `data/trackman_history.csv`를 **gzip 압축**한 `train.csv.gz`(≈74MB), `trackman_history.csv.gz`(≈78MB)를 구글 드라이브에 업로드 (원본 690MB → 압축본 152MB로 업로드가 훨씬 빠르고 안정적입니다. pandas가 `.gz`를 자동으로 읽어서 압축 해제는 따로 안 해도 됩니다)
3. 순차대로 셀 실행 (Runtime > Run all) — 데이터 로드 전에 자동으로 업로드 무결성(용량/행 수)을 확인하는 안전장치 셀이 있습니다

In [ ]:
# 필요한 패키지 설치 (torch는 Colab에 기본 장착됨)
!pip install -q catboost lightgbm

In [ ]:
# 옵션 A: Google Drive 마운트 (data/train.csv.gz, data/trackman_history.csv.gz를 Drive에 미리 올려둔 경우 — 권장)
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/lgaimers9/data'  # 경로는 실제 위치에 맞게 수정 (드라이브에서 폴더 우클릭 > 경로 복사로 확인)

# 옵션 B: 직접 업로드한 경우 (좌측 파일 탭에서 data/ 폴더를 만들고 업로드) — 옵션 A를 쓰면 아래 줄은 주석 처리
# DATA_DIR = '/content/data'

import os
assert os.path.exists(os.path.join(DATA_DIR, 'train.csv.gz')), f'{DATA_DIR}/train.csv.gz 없음 — 업로드 또는 DATA_DIR 경로를 확인하세요'
assert os.path.exists(os.path.join(DATA_DIR, 'trackman_history.csv.gz')), f'{DATA_DIR}/trackman_history.csv.gz 없음'
print('데이터 확인 OK')

In [ ]:

# 안전장치: 무거운 피처 엔지니어링 돌리기 전에 업로드가 제대로 됐는지 빠르게 확인
# (참고: 압축본 기준 train.csv.gz ≈ 74MB, trackman_history.csv.gz ≈ 78MB)
import pandas as pd

_train_path = os.path.join(DATA_DIR, 'train.csv.gz')
_trm_path = os.path.join(DATA_DIR, 'trackman_history.csv.gz')

_train_size_mb = os.path.getsize(_train_path) / (1024 * 1024)
_trm_size_mb = os.path.getsize(_trm_path) / (1024 * 1024)
print(f"train.csv.gz 크기: {_train_size_mb:.1f} MB (정상: 약 74MB)")
print(f"trackman_history.csv.gz 크기: {_trm_size_mb:.1f} MB (정상: 약 78MB)")

_check_df = pd.read_csv(_train_path, usecols=['season'])
print(f"\ntrain.csv 행 수: {len(_check_df):,} (정상: 1,475,092)")
print("시즌별 행 수:")
print(_check_df['season'].value_counts().sort_index())

assert len(_check_df) > 1_400_000, "❌ 행 수가 너무 적습니다 — 업로드가 잘렸을 가능성이 높습니다. Drive에 다시 업로드하세요."
assert 2024 in _check_df['season'].values, "❌ season==2024 데이터가 없습니다 — 파일이 잘렸거나 잘못된 파일입니다."
print("\n✅ 데이터 정상 확인")
del _check_df


In [ ]:
# feature_engineering.py 내용 (open 저장소 / lgaimers9-ensemble-v1 code/train.py와 동일)
import numpy as np
import pandas as pd


def process_trackman_features_safe(df_main, df_trm, is_train_split=True):
    df_main_copy = df_main.copy()
    df_trm_copy = df_trm.copy()

    if is_train_split:
        max_season = df_main_copy['season'].max()
        max_month = df_main_copy[df_main_copy['season'] == max_season]['game_month'].max()
        future_mask = (df_trm_copy['season'] > max_season) | \
                      ((df_trm_copy['season'] == max_season) & (df_trm_copy['game_month'] > max_month))
        df_trm_copy = df_trm_copy[~future_mask].reset_index(drop=True)
        print(f"⏳ [Time Filter] {max_season}년 {max_month}월 이전의 트랙맨 데이터만 잘라내어 피처를 산출합니다.")

    match_cols = [dfc for dfc in df_main_copy.columns if (dfc in df_trm_copy.columns) and dfc != 'row_id']

    grouped = df_trm_copy.groupby(match_cols + ['pitch_type_group', 'auto_pitch_type'])
    grouped_phase1 = grouped[['rel_speed', 'spin_rate', 'induced_vert_break', 'horz_break', 'extension', 'rel_height', 'rel_side', 'zone_speed']].agg(['mean', 'std'])
    grouped_phase2 = grouped_phase1.reset_index()

    std_cols = [col for col in grouped_phase2.columns if 'std' in col]
    grouped_phase2[std_cols] = grouped_phase2[std_cols].fillna(0)
    grouped_phase2.columns = ['_'.join(col).strip('_') for col in grouped_phase2.columns]

    grouped_phase3 = grouped_phase2.drop(columns='auto_pitch_type')
    grouped_phase3 = grouped_phase3.groupby(match_cols + ['pitch_type_group']).agg(['mean'])

    pivoted = grouped_phase3.unstack(level='pitch_type_group')
    pivoted.columns = [f"{col[0]}_{col[1]}_{col[2]}" for col in pivoted.columns]
    tm_final = pivoted.reset_index()
    tm_final = tm_final.fillna(0)

    df_main_copy['top_bottom'] = df_main_copy['top_bottom'].map({'T': 0, 'B': 1}).astype(np.int64)
    tm_final['top_bottom'] = tm_final['top_bottom'].map({'Top': 0, 'Bottom': 1}).astype(np.int64)

    for col in ['batter_hand', 'pitcher_hand']:
        if col in tm_final.columns:
            tm_final[col] = tm_final[col].map({'Left': 1, 'Right': 2}).astype(np.int64)

    tr_final = pd.merge(df_main_copy, tm_final, on=match_cols, how='left')
    new_feature_cols = [col for col in tm_final.columns if col not in match_cols]
    tr_final[new_feature_cols] = tr_final[new_feature_cols].fillna(tr_final[new_feature_cols].mean())

    return tr_final, match_cols


def add_engineered_features(df, league_success_mean):
    df = df.copy()

    df['pitcher_recent1_gap'] = df['asof_pitcher_prev1_game_success_rate'] - df['asof_pitcher_success_rate']
    df['pitcher_recent3_gap'] = df['asof_pitcher_prev3_game_success_rate'] - df['asof_pitcher_success_rate']
    df['pitcher_recent5_gap'] = df['asof_pitcher_prev5_game_success_rate'] - df['asof_pitcher_success_rate']

    df['pitcher_relative_success'] = df['asof_pitcher_success_rate'] - league_success_mean

    df['count_diff'] = df['strikes_before'] - df['balls_before']
    df['is_full_count'] = ((df['balls_before'] == 3) & (df['strikes_before'] == 2)).astype(np.int64)

    df['pitcher_count_advantage_raw'] = df['asof_pitcher_success_rate'] * df['count_diff']
    df['pitcher_count_advantage_rel'] = df['pitcher_relative_success'] * df['count_diff']

    df['pitcher_trend'] = df['asof_pitcher_prev1_game_success_rate'] - df['asof_pitcher_prev5_game_success_rate']
    df['pitcher_consistency'] = df[[
        'asof_pitcher_prev1_game_success_rate',
        'asof_pitcher_prev3_game_success_rate',
        'asof_pitcher_prev5_game_success_rate',
    ]].std(axis=1)

    df['matchup'] = df['asof_pitcher_success_rate'] - df['asof_batter_success_rate']

    pressure_signal = df['li'] * ((df['strikes_before'] >= 2) | (df['balls_before'] >= 3)).astype(np.int64)
    df['count_pressure'] = df['pitcher_relative_success'] * pressure_signal

    return df

In [ ]:
# mlp_model.py 내용 (lgaimers9-ensemble-v1 code/mlp_model.py와 동일, 창현님 검증된 설정 그대로)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

CAT_COLS = [
    "top_bottom", "game_type", "base_state",
    "pitcher_hand", "batter_hand", "pitcher_team_id", "batter_team_id",
]

MAX_EMBED_DIM = 50
MIN_EMBED_DIM = 4
HIDDEN1 = 128
HIDDEN2 = 64
DROPOUT = 0.3
LR = 0.003
WEIGHT_DECAY = 0.01
BATCH_SIZE = 4096
MAX_EPOCHS = 60
PATIENCE = 7
ENSEMBLE_SEEDS = [42, 123, 7, 2024, 99, 555, 31337]


def embed_dim_for_cardinality(cardinality):
    return max(MIN_EMBED_DIM, min(MAX_EMBED_DIM, (cardinality + 1) // 2))


class TabularMLP(nn.Module):
    def __init__(self, num_numeric_feats, cat_dims, embed_dims=None):
        super().__init__()
        if embed_dims is None:
            embed_dims = [embed_dim_for_cardinality(d) for d in cat_dims]
        self.embed_dims = list(embed_dims)
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=dim + 2, embedding_dim=edim) for dim, edim in zip(cat_dims, self.embed_dims)
        ])
        total_input_dim = sum(self.embed_dims) + num_numeric_feats
        self.mlp = nn.Sequential(
            nn.Linear(total_input_dim, HIDDEN1), nn.BatchNorm1d(HIDDEN1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN1, HIDDEN2), nn.BatchNorm1d(HIDDEN2), nn.ReLU(),
            nn.Linear(HIDDEN2, 1),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x_cat, x_num):
        embeds = [emb(x_cat[:, i].long()) for i, emb in enumerate(self.embeddings)]
        x_embed = torch.cat(embeds, dim=1)
        x_all = torch.cat([x_embed, x_num], dim=1)
        return self.sigmoid(self.mlp(x_all)).squeeze(-1)


def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def fit_preprocessing(df, cat_cols, num_cols):
    df = df.copy()
    df[cat_cols] = df[cat_cols].astype(str)
    cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    num_imputer = SimpleImputer(strategy='median')
    num_scaler = StandardScaler()
    df[cat_cols] = cat_encoder.fit_transform(df[cat_cols]) + 1
    df[num_cols] = num_imputer.fit_transform(df[num_cols])
    df[num_cols] = num_scaler.fit_transform(df[num_cols])
    cat_dims = [int(df[col].max() + 1) for col in cat_cols]
    return df, cat_encoder, num_imputer, num_scaler, cat_dims


def apply_preprocessing(df, cat_cols, num_cols, cat_encoder, num_imputer, num_scaler):
    df = df.copy()
    df[cat_cols] = df[cat_cols].astype(str)
    df[cat_cols] = cat_encoder.transform(df[cat_cols]) + 1
    df[num_cols] = num_imputer.transform(df[num_cols])
    df[num_cols] = num_scaler.transform(df[num_cols])
    return df


def to_tensors(df, cat_cols, num_cols, target_col=None):
    X_cat = torch.tensor(df[cat_cols].values.astype(np.float32))
    X_num = torch.tensor(df[num_cols].values.astype(np.float32))
    if target_col is not None:
        y = torch.tensor(df[target_col].values.astype(np.float32))
        return X_cat, X_num, y
    return X_cat, X_num


def compute_bss(preds, y):
    preds = np.asarray(preds, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    brier = ((preds - y) ** 2).mean()
    r = y.mean()
    baseline_brier = r * (1 - r)
    bss = 1.0 - (brier / baseline_brier)
    score = max(0.0, 100000.0 * bss)
    return brier, bss, score


def train_mlp(X_tr_cat, X_tr_num, y_tr, cat_dims, num_numeric_feats, X_val_cat=None, X_val_num=None, y_val=None,
              embed_dims=None, max_epochs=MAX_EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE, lr=LR,
              weight_decay=WEIGHT_DECAY, device=None, verbose=True, seed=42):
    torch.manual_seed(seed)
    device = device or get_device()
    model = TabularMLP(num_numeric_feats=num_numeric_feats, cat_dims=cat_dims, embed_dims=embed_dims).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    dataset = TensorDataset(X_tr_cat, X_tr_num, y_tr)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    has_val = X_val_cat is not None and y_val is not None and len(y_val) > 0
    best_state, best_epoch, best_val_brier, epochs_no_improve = None, 0, float('inf'), 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for batch_cat, batch_num, batch_y in loader:
            batch_cat, batch_num, batch_y = batch_cat.to(device), batch_num.to(device), batch_y.to(device)
            optimizer.zero_grad()
            preds = model(batch_cat, batch_num)
            loss = criterion(preds, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / max(len(loader), 1)
        if has_val:
            model.eval()
            with torch.no_grad():
                val_preds = model(X_val_cat.to(device), X_val_num.to(device)).cpu().numpy()
            val_brier, val_bss, val_score = compute_bss(val_preds, y_val)
            if verbose:
                print(f"Epoch {epoch}/{max_epochs} | Train Loss: {avg_loss:.5f} | Val Brier: {val_brier:.6f} | Val Score: {val_score:.2f}")
            if val_brier < best_val_brier - 1e-9:
                best_val_brier, best_epoch = val_brier, epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    if verbose:
                        print(f"[EarlyStopping] epoch {epoch}에서 종료 (best epoch: {best_epoch})")
                    break
        else:
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()
    return model, best_epoch


def train_ensemble(X_tr_cat, X_tr_num, y_tr, cat_dims, num_numeric_feats, X_val_cat=None, X_val_num=None, y_val=None,
                    seeds=ENSEMBLE_SEEDS, embed_dims=None, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                    batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY, device=None, verbose=True):
    device = device or get_device()
    members = []
    for seed in seeds:
        model, best_epoch = train_mlp(X_tr_cat, X_tr_num, y_tr, cat_dims=cat_dims, num_numeric_feats=num_numeric_feats,
                                        X_val_cat=X_val_cat, X_val_num=X_val_num, y_val=y_val, embed_dims=embed_dims,
                                        max_epochs=max_epochs, patience=patience, batch_size=batch_size, lr=lr,
                                        weight_decay=weight_decay, device=device, verbose=verbose, seed=seed)
        members.append({'state_dict': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 'best_epoch': best_epoch, 'seed': seed})
        print(f"[Ensemble] seed={seed} 학습 완료 (best_epoch={best_epoch})")
    return members


def predict_ensemble(members, cat_dims, num_numeric_feats, embed_dims, X_cat, X_num, device=None):
    device = device or get_device()
    preds_list = []
    with torch.no_grad():
        for member in members:
            model = TabularMLP(num_numeric_feats=num_numeric_feats, cat_dims=cat_dims, embed_dims=embed_dims).to(device)
            model.load_state_dict(member['state_dict'])
            model.eval()
            preds_list.append(model(X_cat.to(device), X_num.to(device)).cpu().numpy())
    return np.mean(preds_list, axis=0)

In [ ]:
# 데이터 로드 + 피처 엔지니어링 (전체 120만 행, 샘플링 없음)
import gc
TARGET = 'control_success'

df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv.gz'))
df_trm = pd.read_csv(os.path.join(DATA_DIR, 'trackman_history.csv.gz'))

# float32 다운캐스트 (메모리 절약용). float64로 해봐도 정확도가 더 낫지 않았고(오히려 낮게 나옴,
# 792.12 -> 776.29) 재학습 자체의 랜덤성(Bayesian bootstrap) 편차로 보여서, 메모리 이득이 확실한
# float32를 다시 씁니다.
for _c in df_trm.select_dtypes(include='float64').columns:
    df_trm[_c] = df_trm[_c].astype('float32')
for _c in df.select_dtypes(include='float64').columns:
    df[_c] = df[_c].astype('float32')

tr_final, _ = process_trackman_features_safe(df, df_trm, is_train_split=True)
train_df = tr_final.dropna(subset=[TARGET]).reset_index(drop=True)

train_mask = train_df['season'] < 2024
val_mask = train_df['season'] == 2024
league_success_mean = train_df.loc[train_mask, TARGET].mean()
train_df = add_engineered_features(train_df, league_success_mean)

for _c in train_df.select_dtypes(include='float64').columns:
    train_df[_c] = train_df[_c].astype('float32')

drop_cols = ['row_id', TARGET]
features = [c for c in train_df.columns if c not in drop_cols]
num_cols = [c for c in features if c not in CAT_COLS]

train_split = train_df.loc[train_mask].reset_index(drop=True)
val_split = train_df.loc[val_mask].reset_index(drop=True)
print(f"train={len(train_split)} val={len(val_split)} features={len(features)} (cat={len(CAT_COLS)}, num={len(num_cols)})")
print(f"train_split 메모리: {train_split.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# 메모리 정리: df/df_trm(원본 raw csv)과 train_df(train_split/val_split으로 이미 분리됨)는 더 필요 없음
# tr_final은 categorical_features_cb 뽑을 때 아래 CatBoost 셀에서 한 번 더 쓰이니 유지
del df, df_trm, train_df
gc.collect()

In [ ]:
# 1) CatBoost 학습 (기존 튜닝된 하이퍼파라미터 그대로, BSS 0.00819 / 818.54를 낸 설정)
# CPU로 고정합니다 (GPU는 BrierScore 미지원이라 5-iteration 단위로만 체크되어 best_iteration이 어긋나고,
# 애초에 이 하이퍼파라미터가 CPU 학습 기준으로 튜닝된 값이라 GPU에서 재현 안 됨 -- score가 800.89로 낮게 나온 원인)
import gc
from catboost import CatBoostClassifier, Pool

categorical_features_cb = [c for c in tr_final.select_dtypes(include='object').columns if c != 'row_id' and c in features]
train_cb = train_split.copy()
val_cb = val_split.copy()
for col in categorical_features_cb:
    train_cb[col] = train_cb[col].astype(str).fillna('missing')
    val_cb[col] = val_cb[col].astype(str).fillna('missing')

CATBOOST_PARAMS = dict(
    iterations=1500, learning_rate=0.05040411253232039, depth=7, l2_leaf_reg=3.14659036827521,
    loss_function='Logloss', border_count=106, random_seed=42, verbose=200,
    random_strength=3.715568024268865, eval_metric='BrierScore', bagging_temperature=0.4609270457436248,
    bootstrap_type='Bayesian', early_stopping_rounds=50, min_data_in_leaf=81,
)
train_pool = Pool(train_cb[features], train_cb[TARGET], cat_features=categorical_features_cb)
val_pool = Pool(val_cb[features], val_cb[TARGET], cat_features=categorical_features_cb)

cat_model = CatBoostClassifier(**CATBOOST_PARAMS)
cat_model.fit(train_pool, eval_set=val_pool)

cat_pred = cat_model.predict_proba(val_pool)[:, 1]
_, cat_bss, cat_score = compute_bss(cat_pred, val_split[TARGET].values)
print(f"CatBoost 단독: score={cat_score:.2f} (참고: 로컬 기준 818.54)")

# 메모리 정리: train_cb/val_cb/Pool 객체는 다 썼으니 바로 해제 (다음 모델 셀에서 램 부족 방지)
del train_cb, val_cb, train_pool, val_pool
gc.collect()

In [ ]:
# 2) MLP 7-seed 앙상블 학습 (GPU면 몇 분 안에 끝납니다)
device = get_device()
print('device:', device)

train_proc, cat_encoder, num_imputer, num_scaler, cat_dims = fit_preprocessing(train_split, CAT_COLS, num_cols)
val_proc = apply_preprocessing(val_split, CAT_COLS, num_cols, cat_encoder, num_imputer, num_scaler)

X_tr_cat, X_tr_num, y_tr = to_tensors(train_proc, CAT_COLS, num_cols, TARGET)
X_val_cat, X_val_num, y_val_t = to_tensors(val_proc, CAT_COLS, num_cols, TARGET)

# train_proc/val_proc는 텐서로 이미 변환됐으니 dataframe 형태는 더 필요 없음 -> 바로 해제
del train_proc, val_proc
gc.collect()

embed_dims = [embed_dim_for_cardinality(d) for d in cat_dims]
members = train_ensemble(X_tr_cat, X_tr_num, y_tr, cat_dims=cat_dims, num_numeric_feats=len(num_cols),
                          X_val_cat=X_val_cat, X_val_num=X_val_num, y_val=y_val_t, embed_dims=embed_dims,
                          device=device, verbose=False)

mlp_pred = predict_ensemble(members, cat_dims, len(num_cols), embed_dims, X_val_cat, X_val_num, device=device)
_, mlp_bss, mlp_score = compute_bss(mlp_pred, val_split[TARGET].values)
print(f"MLP 7-seed 앙상블 (평균): score={mlp_score:.2f} (참고: 로컬 기준 789.58)")

# 진단용: 시드 각각 따로 봤을 때 개별 성능 (블렌드 가중치엔 안 씀, 그냥 참고용으로 보는 것)
print("\n[개별 시드 성능 (진단용, 최종 블렌드 가중치엔 미반영 — 여전히 7개 단순평균을 씁니다)]")
individual_scores = []
for member in members:
    solo_pred = predict_ensemble([member], cat_dims, len(num_cols), embed_dims, X_val_cat, X_val_num, device=device)
    _, _, solo_score = compute_bss(solo_pred, val_split[TARGET].values)
    individual_scores.append(solo_score)
    print(f"  seed={member['seed']:>6} (best_epoch={member['best_epoch']:>2}): score={solo_score:.2f}")
print(f"  개별 시드 range: {min(individual_scores):.2f} ~ {max(individual_scores):.2f} (참고: 보고서 §3.3 기준 715~740대)")
print(f"  7개 단순평균 앙상블: {mlp_score:.2f}  <- 개별 최고치보다도 대체로 더 높게 나오는 게 정상입니다 (분산 감소 효과)")

# X_tr_cat/X_tr_num(120만행 학습텐서)는 학습 다 끝났으니 해제 (X_val_*는 다음 셀들에서 재사용 안 하므로 이것도 해제 가능하지만
# mlp_pred는 이미 뽑아뒀으니 상관없음)
del X_tr_cat, X_tr_num, y_tr, X_val_cat, X_val_num, y_val_t
gc.collect()

In [ ]:
# 3) LightGBM 학습 (open 저장소 tune_lightgbm.py Optuna 40trial 튜닝 완료 결과: 725.87 -> 740.09)
from lightgbm import LGBMClassifier, early_stopping

def lgb_brier_feval(y_true, y_pred):
    return 'brier', float(((y_pred - y_true) ** 2).mean()), False

# model_py311/best_lgb_hparams.json 튜닝 결과 그대로 (Best BSS 0.00740 / score 740.09)
LGB_PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.03217194769265036,
    num_leaves=26,
    max_depth=10,
    subsample=0.6314809868940003,
    colsample_bytree=0.911584046801737,
    reg_lambda=9.631767802618333,
    reg_alpha=0.5110170461186431,
    min_child_samples=42,
    random_state=42, n_jobs=-1, verbose=-1,
)

train_lgb = train_split.copy()
val_lgb = val_split.copy()
for col in categorical_features_cb:
    train_lgb[col] = train_lgb[col].astype(str).fillna('missing').astype('category')
    val_lgb[col] = val_lgb[col].astype(str).fillna('missing').astype('category')

lgb_model = LGBMClassifier(**LGB_PARAMS)
lgb_model.fit(train_lgb[features], train_lgb[TARGET], eval_set=[(val_lgb[features], val_lgb[TARGET])],
               eval_metric=lgb_brier_feval, callbacks=[early_stopping(50, verbose=False)],
               categorical_feature=categorical_features_cb)

lgb_pred = lgb_model.predict_proba(val_lgb[features])[:, 1]
_, lgb_bss, lgb_score = compute_bss(lgb_pred, val_split[TARGET].values)
print(f"LightGBM 단독 (튜닝됨): score={lgb_score:.2f} (참고: 로컬 기준 740.09)")

# 메모리 정리
del train_lgb, val_lgb
gc.collect()

In [ ]:
# 3-b) XGBoost 단독 성능도 참고용으로 확인 (블렌드 가중치 탐색에는 포함 안 함 — LightGBM과 상관계수 0.9612로 너무 겹쳐서)
from xgboost import XGBClassifier

def brier_eval(y_true, y_pred):
    return float(((y_pred - y_true) ** 2).mean())

# open 저장소 tune_xgb.py Optuna 2차 탐색 결과 (model_py311/best_xgb_hparams.json, Trial 012, 로컬 752.70)
XGB_PARAMS = dict(
    max_depth=6, learning_rate=0.014399911854309236, subsample=0.6367420040370018,
    colsample_bytree=0.9920379472108309, reg_lambda=0.06047908057801884, reg_alpha=4.368166043300848,
    min_child_weight=12, gamma=0.008969955448879116, tree_method='hist', enable_categorical=True,
    random_state=42, n_jobs=-1,
)

train_xgb = train_split.copy()
val_xgb = val_split.copy()
for col in categorical_features_cb:
    train_xgb[col] = train_xgb[col].astype(str).fillna('missing').astype('category')
    val_xgb[col] = val_xgb[col].astype(str).fillna('missing').astype('category')

# GPU 런타임이면 XGBoost도 GPU로 (device='cuda'). 실패하면 CPU로 자동 폴백.
try:
    xgb_model = XGBClassifier(**XGB_PARAMS, n_estimators=3000, eval_metric=brier_eval, early_stopping_rounds=50, device='cuda')
    xgb_model.fit(train_xgb[features], train_xgb[TARGET], eval_set=[(val_xgb[features], val_xgb[TARGET])], verbose=False)
except Exception as e:
    print(f"GPU 학습 실패({e}), CPU로 재시도합니다...")
    xgb_model = XGBClassifier(**XGB_PARAMS, n_estimators=3000, eval_metric=brier_eval, early_stopping_rounds=50)
    xgb_model.fit(train_xgb[features], train_xgb[TARGET], eval_set=[(val_xgb[features], val_xgb[TARGET])], verbose=False)

xgb_pred = xgb_model.predict_proba(val_xgb[features])[:, 1]
_, xgb_bss, xgb_score = compute_bss(xgb_pred, val_split[TARGET].values)
print(f"XGBoost 단독 (참고용, 블렌드 미포함): score={xgb_score:.2f} (참고: 로컬 기준 752.70)")
print(f"  XGBoost vs LightGBM 상관계수: {np.corrcoef(xgb_pred, lgb_pred)[0,1]:.4f} (참고: 0.9612 — 그래서 블렌드엔 뺌)")

# 메모리 정리
del train_xgb, val_xgb
gc.collect()

In [ ]:
# 4) 3-way 블렌드 그리드서치 (step=0.02) + 기존 2-way(851.19)와 비교
# XGBoost는 참고용 단독 점수만 위에서 확인했고, LightGBM과 상관계수가 너무 높아(0.9612) 블렌드 가중치 탐색에는 포함하지 않음
y_val = val_split[TARGET].values
print(f"CatBoost vs MLP  상관계수: {np.corrcoef(cat_pred, mlp_pred)[0,1]:.4f}")
print(f"CatBoost vs LGB  상관계수: {np.corrcoef(cat_pred, lgb_pred)[0,1]:.4f}")
print(f"MLP      vs LGB  상관계수: {np.corrcoef(mlp_pred, lgb_pred)[0,1]:.4f}")

best_w, best_bss = None, -np.inf
step = 0.02
for w1 in np.arange(0, 1.0001, step):
    for w2 in np.arange(0, 1.0001 - w1, step):
        w3 = 1 - w1 - w2
        blend = w1 * cat_pred + w2 * mlp_pred + w3 * lgb_pred
        _, bss, _ = compute_bss(blend, y_val)
        if bss > best_bss:
            best_bss, best_w = bss, (round(w1, 2), round(w2, 2), round(w3, 2))
best_score = max(0, 100000 * best_bss)

print('\n' + '=' * 60)
print("개별 모델 단독 성능")
print(f"  CatBoost                : {cat_score:.2f}")
print(f"  MLP (7-seed 앙상블)      : {mlp_score:.2f}")
print(f"  LightGBM (튜닝됨)        : {lgb_score:.2f}")
print(f"  XGBoost (참고용, 블렌드 미포함): {xgb_score:.2f}")
print()
print("블렌드 비교")
print(f"  CatBoost+MLP 2-way (기존, 로컬)   : 851.19 (실제 대시보드 924)")
print(f"  CatBoost+MLP+LightGBM 3-way weights (Cat,MLP,LGB)={best_w}")
print(f"  3-way 블렌드 로컬 점수            : {best_score:.2f}")
print(f"  2-way(851.19) 대비                : {best_score - 851.19:+.2f}")
print('=' * 60)

## 다음 단계 (3-way가 2-way보다 높게 나오면)
1. 이 결과를 팀원들과 공유
2. `dopip.py` 스타일로 전체 데이터(2019~2024)로 3개 모델 재학습 (holdout 없음)
3. `submit/script.py`에 LightGBM 벤더 추가 반영 후 스모크테스트
4. 실제 제출로 대시보드 확인 (기대치: 924보다 높게)